#### SubAgents

In [2]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pprint import pprint

Globals

In [3]:
model = f"bedrock:us.anthropic.claude-opus-4-6-v1"

Tools

In [4]:
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

Creating SubAgents

In [5]:
subagent_1 = create_agent(
    model,
    tools=[square_root]
)

subagent_2 = create_agent(
    model,
    tools=[square]
)

Creating Tools that Invoke SubAgents

In [6]:
@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

Main Agent

In [7]:
main_agent = create_agent(
    model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

Agent Invocation

In [8]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

for m in response['messages']:
    m.pretty_print()

================================ Human Message =================================

What is the square root of 456?
================================== Ai Message ==================================



Let me calculate the square root of 456 for you.
Tool Calls:
  call_subagent_1 (toolu_bdrk_012ZrdpUS2hjdPjxbhyDXr7N)
 Call ID: toolu_bdrk_012ZrdpUS2hjdPjxbhyDXr7N
  Args:
    x: 456
================================= Tool Message =================================
Name: call_subagent_1

The square root of 456.0 is approximately **21.3542**.
================================== Ai Message ==================================

The square root of 456 is approximately **21.3542**.
